# Sindhi-Jev: ablations (Kaggle GPU, 15 h budget)

6 runs x 20k steps (~2.45 GPU h each), defined in `configs/ablations_v1.json`. One commit session runs them one
after another until `SESSION_HOURS` is used (~3 runs), so **2 sessions** cover all 6.
Setup: GPU T4, Internet on, Input: **sindhijev-code** (latest version). For session 2 also add **this notebook's
previous version** as input: finished runs are copied over and skipped, an unfinished one resumes.
Then **Save Version -> Save & Run All (Commit)**.

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
hits = glob.glob('/kaggle/input/**/src/sindhijev/model.py', recursive=True)
if not hits:
    raise SystemExit('sindhijev code not found: add the sindhijev-code dataset under Input')
SRC = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
print('code found at', SRC)
os.chdir(SRC)
!pip -q install uharfbuzz==0.57.0 freetype-py==2.5.1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
SESSION_HOURS = 8.0   # stop starting new work after this (Kaggle limit is longer)
ABL = json.load(open('configs/ablations_v1.json'))
TOTAL = ABL['epochs'] * ABL['steps_per_epoch']
t0 = time.time()

def finished(log):
    if not os.path.exists(log):
        return False
    return any(json.loads(l)['step'] >= TOTAL for l in open(log) if l.strip())

for name, spec in ABL['runs'].items():
    out = f'/kaggle/working/runs/{name}'
    prev = sorted(glob.glob(f'/kaggle/input/**/runs/{name}', recursive=True))
    if prev and finished(f'{prev[0]}/log.jsonl'):
        shutil.copytree(prev[0], out, dirs_exist_ok=True)   # keep finished runs in this version's output
        print(name, 'already finished in a previous session: copied, skipped')
        continue
    left = SESSION_HOURS - (time.time() - t0) / 3600
    if left < 0.5:
        print('session budget used; run the next session for:', name)
        break
    args = ['python', 'scripts/train.py', '--config', ABL['model_config'], '--out', out,
            '--epochs', str(ABL['epochs']), '--steps-per-epoch', str(ABL['steps_per_epoch']),
            '--max-hours', f'{min(left, 3.0):.2f}', '--workers', '4', '--budget-hours', '3.0']
    if prev and os.path.exists(f'{prev[0]}/last.pt') and not os.path.exists(f'{out}/last.pt'):
        args += ['--resume', f'{prev[0]}/last.pt']
        if os.path.exists(f'{prev[0]}/gpu_time.json'):
            os.makedirs(out, exist_ok=True); shutil.copy(f'{prev[0]}/gpu_time.json', out)
    if spec['set']:
        args += ['--set'] + spec['set']
    if spec['data_config']:
        args += ['--data-config', spec['data_config']]
    print('RUN', name, ' '.join(args), flush=True)
    subprocess.run(args, check=True)

In [ ]:
for d in sorted(glob.glob('/kaggle/working/runs/*')):
    g = json.load(open(f'{d}/gpu_time.json')) if os.path.exists(f'{d}/gpu_time.json') else {'sessions': []}
    last = [json.loads(l) for l in open(f'{d}/log.jsonl')][-1] if os.path.exists(f'{d}/log.jsonl') else {}
    print(os.path.basename(d), f"{sum(s['hours'] for s in g['sessions']):.2f} GPU h", 'step', last.get('step'),
          'val_hard', last.get('val_auroc_hard'))